In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [3]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [4]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [5]:
documents_llm = []

for doc in documents:
    if (doc["filename"] == "01-agentic-rag/lessons/01-intro.md"
        or doc["filename"] == "01-agentic-rag/lessons/02-environment.md"
        or doc["filename"] == "01-agentic-rag/lessons/03-rag.md"):
        documents_llm.append(doc)

len(documents_llm)

3

In [6]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()

In [7]:
from pydantic import BaseModel


class QuestionsList(BaseModel):
    questions: list[str]

In [8]:
from tqdm.auto import tqdm

from evaluation_utils import llm_structured_retry

results = []

for doc in tqdm(documents_llm):
    questions, usage = llm_structured_retry(
        client,
        data_gen_instructions,
        doc["content"],
        QuestionsList,
    )

    print(doc["filename"], "input_tokens:", usage.input_tokens)

    results.append({
        "filename": doc["filename"],
        "questions": questions.questions,
        "input_tokens": usage.input_tokens,
    })

  0%|          | 0/3 [00:00<?, ?it/s]

01-agentic-rag/lessons/01-intro.md input_tokens: 943
01-agentic-rag/lessons/02-environment.md input_tokens: 1157
01-agentic-rag/lessons/03-rag.md input_tokens: 1569


In [ ]:
avg_input_tokens = sum(r["input_tokens"] for r in results) / len(results)
avg_input_tokens

In [9]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [10]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [11]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [13]:
from minsearch import Index

index = Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(chunks)

In [18]:
from embedder import Embedder

embedder = Embedder()

contents = [item['content'] for item in chunks]
encoded = embedder.encode_batch(contents)

# Add encoding back to each item
for item, encoding in zip(chunks, encoded):
    item['encoding'] = encoding
    # Now you have: item['content'], item['filename'], item['encoding']

import numpy as np
X = np.array(encoded)

In [19]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, chunks)